In [ ]:
# CELL 0: Install dependencies
!pip install unsloth trl peft accelerate bitsandbytes datasets transformers requests -q
print("All dependencies installed!")

In [ ]:
# CELL 1: Imports and GPU check
import torch
import os, json, re, time, random
import numpy as np
import requests as req
import matplotlib.pyplot as plt
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from transformers import TrainerCallback

print("CUDA:", torch.cuda.is_available())
print("GPU: ", torch.cuda.get_device_name(0))

In [ ]:
# CELL 2: HF Space config and connection test — FILL IN YOUR TOKEN
ENV_BASE_URL = "https://hemankkk-crisis-room.hf.space"
HF_TOKEN     = "YOUR_HF_TOKEN_HERE"   # <-- replace this before running
os.environ["HF_TOKEN"] = HF_TOKEN

# ── Space helper functions ────────────────────────────────────────────────────
def space_reset(difficulty="easy"):
    r = req.post(f"{ENV_BASE_URL}/reset",
                 json={"difficulty": difficulty}, timeout=60)
    r.raise_for_status()
    return r.json()

def space_step(action_type, target=""):
    r = req.post(f"{ENV_BASE_URL}/step",
                 json={"action": {"action_type": action_type, "target": target}},
                 timeout=60)
    r.raise_for_status()
    return r.json()

def normalize(raw):
    return round(max(0.0, min(1.0, float(raw))), 4)

# ── Test connection ───────────────────────────────────────────────────────────
print("Testing HF Space connection...")
try:
    result = space_reset("easy")
    obs    = result["observation"]
    print(f"✅ Space is LIVE!")
    print(f"   Incident:  {obs['message'][:60]}")
    print(f"   Alerts:    {obs['active_alerts'][0]}")
    print(f"   Episode:   {obs['episode_id'][:8]}...")
    
    # Quick step test
    s = space_step("check_logs", "payment-service")
    print(f"   Step test: reward={s['reward']:.3f} done={s['done']}")
    print("\n✅ All systems go — ready to train!")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Check that https://hemankkk-crisis-room.hf.space is Running")

In [ ]:
# CELL 3: Measure baseline score BEFORE training
# This gives us the "before" number for the before/after comparison

NAIVE_ACTIONS = [
    ("easy",   "check_logs",      "payment-service"),
    ("easy",   "restart_service", "api-gateway"),       # wrong — penalized
    ("medium", "check_logs",      "order-service"),
    ("medium", "restart_service", "redis-cluster"),      # lucky guess
    ("hard",   "restart_service", "pricing-service"),   # wrong
    ("hard",   "run_diagnostic",  "dns_check"),          # correct
]

print("Measuring baseline (naive/random actions)...")
baseline_rewards = []

for difficulty, act_type, target in NAIVE_ACTIONS:
    try:
        space_reset(difficulty)
        result = space_step(act_type, target)
        raw  = float(result.get("reward", 0.0))
        norm = normalize(raw)
        baseline_rewards.append(norm)
        print(f"  [{difficulty:6s}] {act_type}:{target[:20]:<20} → {norm:.3f}")
    except Exception as e:
        print(f"  [{difficulty:6s}] failed: {e}")
        baseline_rewards.append(0.0)

BASELINE_SCORE = sum(baseline_rewards) / len(baseline_rewards)
BASELINE_EASY   = sum(baseline_rewards[0:2]) / 2
BASELINE_MEDIUM = sum(baseline_rewards[2:4]) / 2
BASELINE_HARD   = sum(baseline_rewards[4:6]) / 2

print(f"\n📊 BASELINE RESULTS")
print(f"   Easy:    {BASELINE_EASY:.3f}")
print(f"   Medium:  {BASELINE_MEDIUM:.3f}")
print(f"   Hard:    {BASELINE_HARD:.3f}")
print(f"   Average: {BASELINE_SCORE:.3f}")

In [ ]:
# CELL 4: Training dataset — 12 incidents across 3 difficulty levels
INCIDENTS_RAW = [
  # EASY
  {"difficulty":"easy","title":"Payment service returning 500 errors",
   "alerts":"ALERT: payment-service HTTP 500 rate > 40%",
   "status":{"payment-service":"degraded","auth-service":"healthy"},
   "best_action":'{"action_type":"check_logs","target":"payment-service"}',
   "reasoning":"Check logs first to confirm DB pool exhaustion before restarting"},

  {"difficulty":"easy","title":"API gateway returning SSL errors",
   "alerts":"ALERT: api-gateway SSL handshake failures > 95%",
   "status":{"api-gateway":"down","backend-service":"healthy"},
   "best_action":'{"action_type":"run_diagnostic","target":"ssl_check"}',
   "reasoning":"Run ssl_check to confirm certificate expiry before marking resolved"},

  # MEDIUM
  {"difficulty":"medium","title":"Checkout, auth, inventory all degraded",
   "alerts":"ALERT: checkout-service latency p99 > 8s | ALERT: auth-service cache miss 100%",
   "status":{"checkout-service":"degraded","auth-service":"degraded","redis-cluster":"down"},
   "best_action":'{"action_type":"run_diagnostic","target":"memory"}',
   "reasoning":"Multiple services degraded — run memory diagnostic to find redis OOM"},

  {"difficulty":"medium","title":"Order service degrading after deployment",
   "alerts":"ALERT: order-service memory usage > 90%",
   "status":{"order-service":"degraded","payment-service":"healthy"},
   "best_action":'{"action_type":"check_logs","target":"order-service"}',
   "reasoning":"Check logs to confirm memory leak started after v2.4.1 deploy"},

  # HARD
  {"difficulty":"hard","title":"Customers reporting wrong order amounts",
   "alerts":"ALERT: customer complaint rate up 340%",
   "status":{"pricing-service":"healthy","checkout-service":"healthy"},
   "best_action":'{"action_type":"run_diagnostic","target":"data_integrity"}',
   "reasoning":"All services healthy but complaints up — must run data_integrity check"},

  {"difficulty":"hard","title":"Intermittent failures across multiple services",
   "alerts":"ALERT: api-gateway intermittent connection refused 30%",
   "status":{"api-gateway":"degraded","dns-resolver":"degraded"},
   "best_action":'{"action_type":"run_diagnostic","target":"dns_check"}',
   "reasoning":"Intermittent multi-service failure — check DNS"},

  {"difficulty":"hard","title":"Payment service down but root cause is database",
   "alerts":"ALERT: payment-service HTTP 500 rate > 90%",
   "status":{"payment-service":"degraded","postgres-db":"degraded","auth-service":"healthy"},
   "best_action":'{"action_type":"check_logs","target":"postgres-db"}',
   "reasoning":"Root cause is postgres-db not payment-service — check DB logs first"},

  {"difficulty":"hard","title":"Everything looks healthy but users cant login",
   "alerts":"ALERT: user login failure rate 78%",
   "status":{"auth-service":"healthy","session-store":"healthy","api-gateway":"healthy"},
   "best_action":'{"action_type":"run_diagnostic","target":"dns_check"}',
   "reasoning":"All services healthy — hidden DNS issue causing login failures"},

  {"difficulty":"hard","title":"High memory alert but do not restart yet",
   "alerts":"ALERT: order-service memory > 95%",
   "status":{"order-service":"degraded","redis":"healthy"},
   "best_action":'{"action_type":"run_diagnostic","target":"memory"}',
   "reasoning":"Must run memory diagnostic first before restarting"},

  {"difficulty":"hard","title":"Three services down simultaneously",
   "alerts":"ALERT: checkout down | ALERT: inventory down | ALERT: pricing down",
   "status":{"checkout-service":"down","inventory-service":"down",
              "pricing-service":"down","shared-cache":"degraded"},
   "best_action":'{"action_type":"check_logs","target":"shared-cache"}',
   "reasoning":"3 services down at once means shared dependency — check shared-cache"},

  {"difficulty":"hard","title":"Rollback caused more issues than deployment",
   "alerts":"ALERT: api-gateway 503 after rollback attempt",
   "status":{"api-gateway":"down","backend-v1":"degraded","backend-v2":"stopped"},
   "best_action":'{"action_type":"run_diagnostic","target":"data_integrity"}',
   "reasoning":"Rollback made things worse — run data_integrity to find corruption"},

  {"difficulty":"hard","title":"CPU normal memory normal but latency spiking",
   "alerts":"ALERT: api-gateway p99 latency > 15s",
   "status":{"api-gateway":"degraded","upstream-service":"healthy","db":"healthy"},
   "best_action":'{"action_type":"run_diagnostic","target":"dns_check"}',
   "reasoning":"No obvious resource issue — hidden DNS resolution delay causing latency spike"},
]

SYSTEM_PROMPT = """You are an AI Site Reliability Engineer responding to a live production incident.
Investigate first, then fix, then notify the team.
Available actions: check_logs, run_diagnostic, restart_service, rollback_deployment, scale_up, notify_team, escalate, mark_resolved
Respond ONLY with a JSON object: {"action_type": "...", "target": "..."}"""

def make_prompt(inc):
    return f"""{SYSTEM_PROMPT}

INCIDENT: {inc['title']}
DIFFICULTY: {inc['difficulty'].upper()}
ACTIVE ALERTS: {inc['alerts']}
SERVICE STATUS: {json.dumps(inc['status'])}
REASONING HINT: {inc['reasoning']}

What is your next action?"""

dataset = Dataset.from_list([
    {"prompt": make_prompt(inc), "completion": inc["best_action"]}
    for inc in INCIDENTS_RAW
])

print(f"Dataset ready: {len(dataset)} incidents")
print(f"  Easy:   {sum(1 for i in INCIDENTS_RAW if i['difficulty']=='easy')}")
print(f"  Medium: {sum(1 for i in INCIDENTS_RAW if i['difficulty']=='medium')}")
print(f"  Hard:   {sum(1 for i in INCIDENTS_RAW if i['difficulty']=='hard')}")

In [ ]:
# CELL 5: parse_action + all reward functions using live HF Space

def parse_action(text):
    """Extract JSON action from model output"""
    try:
        clean = text.strip()
        if "```" in clean:
            clean = clean.split("```")[1]
            if clean.startswith("json"):
                clean = clean[4:]
        match = re.search(r'\{[^}]+\}', clean)
        if match:
            return json.loads(match.group())
    except:
        pass
    return {"action_type": "check_logs", "target": ""}


def reward_environment_score(completions, prompts=None, **kwargs):
    """Run FULL EPISODES on live HF Space — maximum reward ceiling"""
    rewards = []
    difficulties = ["easy", "medium", "hard", "easy", "medium", "hard"]

    for i, completion in enumerate(completions):
        try:
            action     = parse_action(completion)
            difficulty = difficulties[i % len(difficulties)]

            reset_result = space_reset(difficulty)
            obs          = reset_result["observation"]
            ep_rewards   = []

            for step in range(obs["max_steps"]):
                if step == 0:
                    act_type   = action.get("action_type", "check_logs")
                    act_target = action.get("target", "")
                else:
                    taken      = obs.get("actions_taken", [])
                    logged     = any("check_logs"      in a for a in taken)
                    diagnosed  = any("run_diagnostic"  in a for a in taken)
                    notified   = any("notify_team"     in a for a in taken)
                    root_found = obs.get("root_cause_found", False)

                    if not logged:
                        degraded   = [k for k,v in obs.get("service_status",{}).items()
                                      if v != "healthy"]
                        act_type   = "check_logs"
                        act_target = degraded[0] if degraded else "api-gateway"
                    elif not diagnosed:
                        act_type   = "run_diagnostic"
                        act_target = random.choice(["memory","db_connections","dns_check"])
                    elif not notified:
                        act_type   = "notify_team"
                        act_target = "investigating root cause, update in 5 min"
                    elif root_found:
                        act_type   = "mark_resolved"
                        act_target = "root_cause_identified_and_fixed"
                    else:
                        degraded   = [k for k,v in obs.get("service_status",{}).items()
                                      if v != "healthy"]
                        act_type   = "restart_service"
                        act_target = degraded[0] if degraded else "payment-service"

                result = space_step(act_type, act_target)
                obs    = result["observation"]
                ep_rewards.append(float(result.get("reward", 0.0)))

                if result.get("done"):
                    break

            best_raw   = max(ep_rewards) if ep_rewards else 0.0
            normalized = normalize(best_raw)
            rewards.append(normalized)

        except Exception as e:
            print(f"[WARN] episode {i} failed: {e}")
            rewards.append(0.1)

    return rewards


def reward_valid_json(completions, **kwargs):
    """Reward valid JSON with correct action_type and non-empty target"""
    valid = ["check_logs","run_diagnostic","restart_service",
             "rollback_deployment","scale_up","notify_team","escalate","mark_resolved"]
    rewards = []
    for c in completions:
        try:
            a = parse_action(c)
            if a.get("action_type") in valid and a.get("target","").strip():
                rewards.append(1.0)
            elif a.get("action_type") in valid:
                rewards.append(0.6)
            else:
                rewards.append(0.1)
        except:
            rewards.append(0.0)
    return rewards


def reward_investigate_first(completions, prompts=None, **kwargs):
    """Reward investigation actions with meaningful targets"""
    meaningful = ["payment-service","api-gateway","redis-cluster","order-service",
                  "dns_check","data_integrity","memory","db_connections","ssl_check",
                  "postgres-db","shared-cache","pricing-service"]
    rewards = []
    for c in completions:
        a   = parse_action(c)
        act = a.get("action_type","")
        tgt = a.get("target","")
        if act in ["check_logs","run_diagnostic"]:
            rewards.append(1.0 if any(m in tgt for m in meaningful) else 0.7)
        elif act in ["notify_team","escalate"]:
            rewards.append(0.5)
        elif act in ["restart_service","rollback_deployment","scale_up"]:
            rewards.append(0.2)
        else:
            rewards.append(0.3)
    return rewards

print("✅ parse_action ready")
print("✅ reward_environment_score — full episodes on live HF Space")
print("✅ reward_valid_json        — rewards specific meaningful targets")
print("✅ reward_investigate_first — rewards investigation before action")

In [ ]:
# CELL 6: Load model with LoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 2048,
    load_in_4bit   = True,
    dtype          = None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r                        = 16,
    target_modules           = ["q_proj","k_proj","v_proj","o_proj",
                                 "gate_proj","up_proj","down_proj"],
    lora_alpha               = 16,
    lora_dropout             = 0,
    bias                     = "none",
    use_gradient_checkpointing = "unsloth",
    random_state             = 42,
)
print("✅ Model + LoRA ready!")
print(f"   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# CELL 7: Reward tracker callback for clean plotting

class RewardTracker(TrainerCallback):
    def __init__(self):
        self.steps             = []
        self.raw_rewards       = []
        self.normalized_rewards = []
        self.losses            = []
        self.loss_steps        = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        step = state.global_step
        if "reward" in logs:
            raw  = float(logs["reward"])
            norm = raw / 3.0          # 3 reward functions summed → normalize to 0-1
            norm = max(0.0, min(1.0, norm))
            self.steps.append(step)
            self.raw_rewards.append(raw)
            self.normalized_rewards.append(norm)
        if "loss" in logs:
            self.loss_steps.append(step)
            self.losses.append(float(logs["loss"]))

reward_tracker = RewardTracker()
print("✅ RewardTracker callback ready")

In [ ]:
# CELL 8: GRPO Training — connects to live HF Space
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir                  = "crisis_room_model_v4",
    learning_rate               = 2e-5,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_train_epochs            = 8,
    max_prompt_length           = 512,
    max_completion_length       = 128,
    num_generations             = 4,
    temperature                 = 0.9,
    logging_steps               = 1,
    save_steps                  = 20,
    report_to                   = "none",
    remove_unused_columns       = False,
)

trainer = GRPOTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = dataset,
    reward_funcs     = [
        reward_environment_score,
        reward_valid_json,
        reward_investigate_first,
    ],
    processing_class = tokenizer,
    callbacks        = [reward_tracker],
)

print("Starting GRPO training against live HF Space...")
print(f"Space URL: {ENV_BASE_URL}")
print(f"Dataset:   {len(dataset)} incidents | Epochs: 8 | Est time: 20-30 min")
start = time.time()
trainer.train()
elapsed = (time.time() - start) / 60
print(f"\n✅ Training complete in {elapsed:.1f} min")
print(f"   Steps logged:   {len(reward_tracker.steps)}")
print(f"   Rewards logged: {len(reward_tracker.normalized_rewards)}")

In [ ]:
# CELL 9: Measure trained model score against live Space

FastLanguageModel.for_inference(model)

def run_trained_episode(difficulty="easy"):
    """Run trained model for one full episode against live Space"""
    reset_result  = space_reset(difficulty)
    obs           = reset_result["observation"]
    history       = []
    ep_rewards    = []

    for step in range(obs["max_steps"]):
        prompt = f"""{SYSTEM_PROMPT}

INCIDENT: {obs['message']}
ALERTS: {obs['active_alerts']}
STATUS: {json.dumps(obs['service_status'])}
LOG: {obs.get('log_output') or 'None yet'}
HISTORY: {history[-3:]}

Next action (JSON only):"""

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        out    = model.generate(
            **inputs,
            max_new_tokens      = 60,
            temperature         = 0.3,
            do_sample           = True,
            pad_token_id        = tokenizer.eos_token_id,
        )
        response = tokenizer.decode(out[0], skip_special_tokens=True)
        response = response[len(prompt):].strip()

        action      = parse_action(response)
        step_result = space_step(
            action.get("action_type", "check_logs"),
            action.get("target", "")
        )
        obs    = step_result["observation"]
        raw    = float(step_result.get("reward", 0.0))
        norm   = normalize(raw)
        ep_rewards.append(norm)
        history.append(f"Step {step+1}: {action.get('action_type')}:{action.get('target')}")

        if step_result.get("done"):
            break

    return max(ep_rewards) if ep_rewards else 0.0

print("Measuring trained model scores...")
trained_easy   = run_trained_episode("easy")
print(f"  Easy:   {trained_easy:.3f}")
trained_medium = run_trained_episode("medium")
print(f"  Medium: {trained_medium:.3f}")
trained_hard   = run_trained_episode("hard")
print(f"  Hard:   {trained_hard:.3f}")

TRAINED_SCORE  = (trained_easy + trained_medium + trained_hard) / 3
TRAINED_EASY   = trained_easy
TRAINED_MEDIUM = trained_medium
TRAINED_HARD   = trained_hard

improvement = ((TRAINED_SCORE - BASELINE_SCORE) / max(BASELINE_SCORE, 0.001)) * 100
print(f"\n📊 TRAINED SCORE:  {TRAINED_SCORE:.3f}")
print(f"📊 BASELINE SCORE: {BASELINE_SCORE:.3f}")
print(f"📊 IMPROVEMENT:    +{improvement:.1f}%")

In [ ]:
# CELL 10: Generate all 4 training charts

norm_rewards = reward_tracker.normalized_rewards
steps        = reward_tracker.steps
losses       = reward_tracker.losses
loss_steps   = reward_tracker.loss_steps

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.patch.set_facecolor('#0f1117')
for ax in axes.flat:
    ax.set_facecolor('#1a1d2e')
    ax.tick_params(colors='#aaa', labelsize=10)
    for spine in ax.spines.values():
        spine.set_color('#333')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.15, color='#444')

# ── Top Left: Training Loss ────────────────────────────────────────────────────
if losses:
    axes[0,0].plot(loss_steps, losses, color='#378ADD',
                   linewidth=2, marker='o', markersize=3)
    axes[0,0].fill_between(loss_steps, losses, alpha=0.15, color='#378ADD')
    axes[0,0].set_title('Training Loss', color='white', fontsize=13, pad=10)
    axes[0,0].set_xlabel('Step', color='#aaa', fontsize=10)
    axes[0,0].set_ylabel('Loss', color='#aaa', fontsize=10)
else:
    axes[0,0].text(0.5, 0.5, 'No loss data', color='white',
                   ha='center', va='center', transform=axes[0,0].transAxes)

# ── Top Right: Normalized Reward 0-1 with baseline + smoothed ─────────────────
if norm_rewards:
    window   = min(5, len(norm_rewards))
    smoothed = np.convolve(norm_rewards, np.ones(window)/window, mode='valid')
    sm_steps = steps[window-1:]

    axes[0,1].plot(steps, norm_rewards, color='#1D9E75',
                   linewidth=1, alpha=0.35, label='Raw reward')
    axes[0,1].plot(sm_steps, smoothed, color='#1D9E75',
                   linewidth=2.5, label=f'Smoothed ({window}-step avg)')
    axes[0,1].fill_between(sm_steps, smoothed, alpha=0.2, color='#1D9E75')
    axes[0,1].axhline(y=BASELINE_SCORE, color='#ff6b6b',
                       linestyle='--', linewidth=1.5,
                       label=f'Baseline ({BASELINE_SCORE:.3f})')
    axes[0,1].set_ylim(0, 1.0)
    axes[0,1].set_title('Environment Reward (Normalized 0-1)',
                         color='white', fontsize=13, pad=10)
    axes[0,1].set_xlabel('Training Step', color='#aaa', fontsize=10)
    axes[0,1].set_ylabel('Reward (0-1)', color='#aaa', fontsize=10)
    axes[0,1].legend(facecolor='#1a1d2e', labelcolor='white', fontsize=9)
else:
    axes[0,1].text(0.5, 0.5, 'No reward data', color='white',
                   ha='center', va='center', transform=axes[0,1].transAxes)

# ── Bottom Left: Before vs After bar chart ────────────────────────────────────
categories    = ['Easy', 'Medium', 'Hard', 'Average']
before_vals   = [BASELINE_EASY, BASELINE_MEDIUM, BASELINE_HARD, BASELINE_SCORE]
after_vals    = [TRAINED_EASY,  TRAINED_MEDIUM,  TRAINED_HARD,  TRAINED_SCORE]

x     = np.arange(len(categories))
width = 0.35

bars1 = axes[1,0].bar(x - width/2, before_vals, width,
                       label='Before Training', color='#ff6b6b',
                       alpha=0.85, edgecolor='#222')
bars2 = axes[1,0].bar(x + width/2, after_vals,  width,
                       label='After Training',  color='#1D9E75',
                       alpha=0.85, edgecolor='#222')

for bar in bars1:
    h = bar.get_height()
    axes[1,0].text(bar.get_x() + bar.get_width()/2., h + 0.01,
                   f'{h:.2f}', ha='center', va='bottom',
                   color='white', fontsize=9, fontweight='bold')
for bar in bars2:
    h = bar.get_height()
    axes[1,0].text(bar.get_x() + bar.get_width()/2., h + 0.01,
                   f'{h:.2f}', ha='center', va='bottom',
                   color='white', fontsize=9, fontweight='bold')

axes[1,0].set_xticks(x)
axes[1,0].set_xticklabels(categories, color='white', fontsize=11)
axes[1,0].set_ylim(0, 1.1)
axes[1,0].set_title('Before vs After Training', color='white', fontsize=13, pad=10)
axes[1,0].set_ylabel('Normalized Reward (0-1)', color='#aaa', fontsize=10)
axes[1,0].legend(facecolor='#1a1d2e', labelcolor='white', fontsize=9)

improvement = ((TRAINED_SCORE - BASELINE_SCORE) / max(BASELINE_SCORE, 0.001)) * 100
axes[1,0].text(0.5, 0.93,
               f'Overall Improvement: +{improvement:.1f}%',
               transform=axes[1,0].transAxes, ha='center',
               color='#1D9E75', fontsize=12, fontweight='bold')

# ── Bottom Right: Reward breakdown stacked area ────────────────────────────────
if norm_rewards:
    raw3     = reward_tracker.raw_rewards
    env_c    = [min(1.0, r / 3.0 * 1.0)         for r in raw3]
    json_c   = [min(0.34, r / 3.0 * 0.34)        for r in raw3]
    invest_c = [min(0.33, r / 3.0 * 0.33)        for r in raw3]
    sp       = list(range(len(raw3)))

    axes[1,1].stackplot(sp,
                        invest_c, json_c, env_c,
                        labels=['Investigate First',
                                'Valid JSON',
                                'Environment Score'],
                        colors=['#f4a261','#378ADD','#1D9E75'],
                        alpha=0.85)
    axes[1,1].set_ylim(0, 1.0)
    axes[1,1].set_title('Reward Breakdown by Component',
                          color='white', fontsize=13, pad=10)
    axes[1,1].set_xlabel('Training Step', color='#aaa', fontsize=10)
    axes[1,1].set_ylabel('Reward (0-1)', color='#aaa', fontsize=10)
    axes[1,1].legend(facecolor='#1a1d2e', labelcolor='white',
                      fontsize=9, loc='lower right')
else:
    axes[1,1].text(0.5, 0.5, 'No reward data', color='white',
                   ha='center', va='center', transform=axes[1,1].transAxes)

fig.suptitle(
    f'Crisis Room SRE Agent — GRPO Training Results\n'
    f'Baseline: {BASELINE_SCORE:.3f}  →  Trained: {TRAINED_SCORE:.3f}  '
    f'(+{improvement:.1f}% improvement)',
    color='white', fontsize=14, y=1.01
)

plt.tight_layout()
plt.savefig("crisis_room_training_results.png", dpi=150,
            bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("✅ Chart saved as crisis_room_training_results.png")

print(f"\n{'='*50}")
print(f"FINAL RESULTS SUMMARY")
print(f"{'='*50}")
print(f"Baseline → Trained")
print(f"  Easy:    {BASELINE_EASY:.3f} → {TRAINED_EASY:.3f}")
print(f"  Medium:  {BASELINE_MEDIUM:.3f} → {TRAINED_MEDIUM:.3f}")
print(f"  Hard:    {BASELINE_HARD:.3f} → {TRAINED_HARD:.3f}")
print(f"  Average: {BASELINE_SCORE:.3f} → {TRAINED_SCORE:.3f}  (+{improvement:.1f}%)")
print(f"{'='*50}")

In [ ]:
# CELL 11: Save model and push everything to HuggingFace
from huggingface_hub import HfApi, login

login(token=HF_TOKEN)
api = HfApi()

# Save model locally
model.save_pretrained("crisis_room_agent_final")
tokenizer.save_pretrained("crisis_room_agent_final")
print("✅ Model saved locally!")

# Push trained model to HF Hub
print("Pushing model to HF Hub...")
api.upload_folder(
    folder_path = "crisis_room_agent_final",
    repo_id     = "DebugThugs/crisis-room-model",
    repo_type   = "model",
)
print("✅ Model pushed → https://huggingface.co/DebugThugs/crisis-room-model")

# Push reward chart to Space repo
api.upload_file(
    path_or_fileobj = "crisis_room_training_results.png",
    path_in_repo    = "reward_curve.png",
    repo_id         = "DebugThugs/crisis-room",
    repo_type       = "space",
)
print("✅ Chart pushed → https://huggingface.co/spaces/DebugThugs/crisis-room")

print("\n🎉 SUBMISSION COMPLETE!")
print(f"  Space:  https://huggingface.co/spaces/DebugThugs/crisis-room")
print(f"  Model:  https://huggingface.co/DebugThugs/crisis-room-model")
print(f"  Score:  {BASELINE_SCORE:.3f} → {TRAINED_SCORE:.3f} (+{improvement:.1f}%)")